1. SHAP analizinde tespit ettiğimiz, modelin ezberlemesine (Data Leakage) sebep olan o üç sütunu hem eğitim hem de test setinden silme

In [1]:
pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 2.4 MB/s eta 0:00:01
   ------------------- -------------------- 1.0/2.1 MB 2.1 MB/s eta 0:00:01
   ------------------- -------------------- 1.0/2.1 MB 2.1 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 1.3 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.4 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.3 MB/s  0:00:01

   ---------------------------------------- 0/7 [PyYAML]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----------- ---------------------------- 2/7 [greenlet]
   ----------- ---------------------------- 2/7 [greenlet]
   ---------------------- ----------------- 4/7

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Veriler RAM'e alınıyor...")
X_train = pd.read_parquet('../data/splits/X_train.parquet')
X_test = pd.read_parquet('../data/splits/X_test.parquet')
y_train = pd.read_parquet('../data/splits/y_train.parquet')
y_test = pd.read_parquet('../data/splits/y_test.parquet')

# SHAP analizinde yakaladığımız ezberci (leaky) sütunlar
leaky_features = ['Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Pkt Len Max']

print("Modeli kandıran özellikler siliniyor...")
X_train.drop(columns=leaky_features, inplace=True, errors='ignore')
X_test.drop(columns=leaky_features, inplace=True, errors='ignore')

print(f"Güncel Özellik (Sütun) Sayısı: {X_train.shape[1]}")

c:\Users\stasd\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Veriler RAM'e alınıyor...
Modeli kandıran özellikler siliniyor...
Güncel Özellik (Sütun) Sayısı: 66


2. Optuna ile Hyperparameter Tuning

In [2]:
print("\nOptuna için geçici Tuning seti oluşturuluyor...")
# Optuna'nın çalışması için Train setini kendi içinde %80-%20 bölüyoruz
X_tune_train, X_tune_val, y_tune_train, y_tune_val = train_test_split(
    X_train, y_train['Label'], test_size=0.2, random_state=42, stratify=y_train['Label']
)

def objective(trial):
    # Optuna'nın deneyeceği parametre uzayları
    param = {
        'scale_pos_weight': 9.079, # AŞAMA 3'te bulduğumuz değişmez değer
        'tree_method': 'hist',     # RAM dostu algoritma
        'n_jobs': -1,              # Tüm CPU çekirdeklerini kullan
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 6), # Detaylara inmesini engelle
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9)
    }
    
    # Modeli bu geçici parametrelerle kur
    model = XGBClassifier(**param)
    
    # Tuning eğitim setinde eğit
    model.fit(X_tune_train, y_tune_train, verbose=False)
    
    # Tuning doğrulama setinde tahmin yap
    y_pred = model.predict(X_tune_val)
    
    recall = recall_score(y_tune_val, y_pred)
    return recall

print("Hiperparametre optimizasyonu başlıyor...")

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # Süreyi kısaltmak için 10 deneme yapıyoruz

print("\n--- EN İYİ PARAMETRELER BULUNDU ---")
best_params = study.best_params
print(best_params)


Optuna için geçici Tuning seti oluşturuluyor...


[I 2026-06-04 21:17:08,064] A new study created in memory with name: no-name-d8e33599-194c-4ca7-85a4-af84543bce47


Hiperparametre optimizasyonu başlıyor...


[I 2026-06-04 21:17:50,394] Trial 0 finished with value: 0.9959346497882386 and parameters: {'n_estimators': 91, 'max_depth': 6, 'learning_rate': 0.022089573337608987, 'subsample': 0.8942618526596484, 'colsample_bytree': 0.7977377576209347}. Best is trial 0 with value: 0.9959346497882386.
[I 2026-06-04 21:18:33,293] Trial 1 finished with value: 0.9958116348682862 and parameters: {'n_estimators': 92, 'max_depth': 6, 'learning_rate': 0.01530215904429427, 'subsample': 0.7565893682888251, 'colsample_bytree': 0.7163358700268125}. Best is trial 0 with value: 0.9959346497882386.
[I 2026-06-04 21:19:26,474] Trial 2 finished with value: 0.992279349309652 and parameters: {'n_estimators': 137, 'max_depth': 4, 'learning_rate': 0.013709156938662313, 'subsample': 0.7151891324058067, 'colsample_bytree': 0.837526889207673}. Best is trial 0 with value: 0.9959346497882386.
[I 2026-06-04 21:20:08,625] Trial 3 finished with value: 0.9955538893217192 and parameters: {'n_estimators': 118, 'max_depth': 5, 'l


--- EN İYİ PARAMETRELER BULUNDU ---
{'n_estimators': 125, 'max_depth': 4, 'learning_rate': 0.05652769278716449, 'subsample': 0.7481862212017223, 'colsample_bytree': 0.7684664229989239}
